# motor outcomes analysis

relationship between VTA-ROI overlap and MDS-UPDRS III motor outcomes.

**outcomes** (initial → final, postop conditions given as medication/stimulation state)

| column | contrast | role |
|---|---|---|
| `MDSUPDRS_G` | postop OFF med/OFF stim → postop OFF med/ON stim | **primary** |
| `MDSUPDRS_A` | preop OFF med → postop ON med/ON stim | secondary |
| `MDSUPDRS_C` | preop OFF med → postop OFF med/ON stim | secondary |
| `MDSUPDRS_E` | postop ON med/OFF stim → postop ON med/ON stim | secondary |
| `MDSUPDRS_F` | postop OFF med/OFF stim → postop ON med/ON stim | secondary |

`MDSUPDRS_G` is the primary endpoint since it is referenced to a postop reading and so is unaffected by disease progression, microlesion effects and medication changes.

**predictors:** VTA overlap with STN and subdivisions (STN patients), GPi and subdivisions (GPi patients). left, right and cross-hemisphere average of each. RN is excluded (near-zero overlap in most patients).

**covariates:** age, sex, laterality.

## 0. setup

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import seaborn as sns
from scipy import stats
from scipy.stats import shapiro, pearsonr, spearmanr
import statsmodels.formula.api as smf
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')

DATA_FILE = '<path to outcomes_data.xlsx>'
FIG_DIR = 'figures_motor'
OUT_CSV = 'motor_results_all.csv'

# plot style
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')
sns.set_palette('Set2')

os.makedirs(FIG_DIR, exist_ok=True)


# avg_GPi_premotor -> avg. GPi premotor
def clean_label(col):
    parts = col.split('_')
    if len(parts) >= 2:
        return parts[0] + '. ' + ' '.join(parts[1:])
    return col


def save_fig(fig, filename):
    path = os.path.join(FIG_DIR, filename + '.tif')
    fig.savefig(path, format='tiff', dpi=300, bbox_inches='tight')
    print(f'  saved: {path}')

## 1. load data

In [ ]:
# optional subject exclusions, empty by default
EXCLUDE_SUBJECTS = set()

df = pd.read_excel(DATA_FILE)
df = df.rename(columns={'id': 'PatientID'})

if EXCLUDE_SUBJECTS:
    n0 = len(df)
    df = df[~df['PatientID'].isin(EXCLUDE_SUBJECTS)]
    print(f'Excluded {n0 - len(df)} subject(s): {sorted(EXCLUDE_SUBJECTS)}')

# cross-hemisphere average, (L + R) / 2 with a non-implanted hemisphere as 0
roi_names = sorted({c[2:] for c in df.columns if c[:2] in ('L_', 'R_')})
for roi in roi_names:
    l = df[f'L_{roi}'] if f'L_{roi}' in df.columns else 0
    r = df[f'R_{roi}'] if f'R_{roi}' in df.columns else 0
    df[f'avg_{roi}'] = (pd.Series(l, index=df.index).fillna(0)
                        + pd.Series(r, index=df.index).fillna(0)) / 2
print(f'Derived {len(roi_names)} avg_ predictors: {roi_names}')

print(f'\nDataset shape: {df.shape}')
print(f'\nTarget distribution:\n{df["Target"].value_counts()}')
print(f'\nLaterality distribution:\n{df["Laterality"].value_counts()}')
print(f'\nSex distribution:\n{df["Sex"].value_counts()}')
print(f'\nAge: mean={df["Age"].mean():.1f}, sd={df["Age"].std():.1f}, '
      f'range=[{df["Age"].min():.0f}, {df["Age"].max():.0f}]')

In [ ]:
# outcome definitions, primary first
SCORES = {
    'MDSUPDRS_G': dict(initial='postOFF/OFF', final='postOFF/ON',  role='primary',
                       desc='postop off-med/off-stim → postop off-med/on-stim'),
    'MDSUPDRS_A': dict(initial='preOFF',      final='postON/ON',   role='secondary',
                       desc='preop off-med → postop on-med/on-stim'),
    'MDSUPDRS_C': dict(initial='preOFF',      final='postOFF/ON',  role='secondary',
                       desc='preop off-med → postop off-med/on-stim'),
    'MDSUPDRS_E': dict(initial='postON/OFF',  final='postON/ON',   role='secondary',
                       desc='postop on-med/off-stim → postop on-med/on-stim'),
    'MDSUPDRS_F': dict(initial='postOFF/OFF', final='postON/ON',   role='secondary',
                       desc='postop off-med/off-stim → postop on-med/on-stim'),
}

OUTCOMES_ALL = list(SCORES)
missing = [c for c in OUTCOMES_ALL if c not in df.columns]
OUTCOMES = [c for c in OUTCOMES_ALL if c in df.columns]

if missing:
    print(f'not present in data file, skipped: {missing}')
print(f'Analysing {len(OUTCOMES)} outcome(s): {OUTCOMES}')

# short label used in tables and figure titles
OUTCOME_LABELS = {k: f"{v['initial']} → {v['final']}" for k, v in SCORES.items()}
PRIMARY = [k for k, v in SCORES.items() if v['role'] == 'primary' and k in OUTCOMES]

for k in OUTCOMES:
    star = '  <- primary endpoint' if k in PRIMARY else ''
    print(f'  {k}: {OUTCOME_LABELS[k]}{star}')

# predictor sets, RN excluded
STN_PREDICTORS_L   = ['L_STN', 'L_STN_associative', 'L_STN_limbic', 'L_STN_motor']
STN_PREDICTORS_R   = ['R_STN', 'R_STN_associative', 'R_STN_limbic', 'R_STN_motor']
STN_PREDICTORS_AVG = ['avg_STN', 'avg_STN_associative', 'avg_STN_limbic', 'avg_STN_motor']

GPi_PREDICTORS_L   = ['L_GPe', 'L_GPi', 'L_GPi_occipital', 'L_GPi_postparietal', 'L_GPi_prefrontal',
                       'L_GPi_premotor', 'L_GPi_primarymotor', 'L_GPi_sensorimotor', 'L_GPi_sensory']
GPi_PREDICTORS_R   = ['R_GPe', 'R_GPi', 'R_GPi_occipital', 'R_GPi_postparietal', 'R_GPi_prefrontal',
                       'R_GPi_premotor', 'R_GPi_primarymotor', 'R_GPi_sensorimotor', 'R_GPi_sensory']
GPi_PREDICTORS_AVG = ['avg_GPe', 'avg_GPi', 'avg_GPi_occipital', 'avg_GPi_postparietal', 'avg_GPi_prefrontal',
                       'avg_GPi_premotor', 'avg_GPi_primarymotor', 'avg_GPi_sensorimotor', 'avg_GPi_sensory']

# drop predictors with < MIN_NONZERO patients with any overlap in the motor subgroup
# (depends only on the predictor, not the outcome)
MIN_NONZERO = 5

def drop_unusable(predictors, sub, label):
    keep, dropped = [], []
    for p in predictors:
        if p not in sub.columns:
            continue
        x = sub[p].dropna()
        nz = int((x != 0).sum())
        (keep if nz >= MIN_NONZERO else dropped).append((p, nz, len(x)))
    if dropped:
        print(f'  {label}: dropped {len(dropped)} predictor(s) with < {MIN_NONZERO} non-zero patients')
        for p, nz, n in dropped:
            print(f'      {p}: {nz}/{n} patients with any overlap')
    return [p for p, _, _ in keep]

_motor = df[df[OUTCOMES].notna().any(axis=1)]
_stn, _gpi = _motor[_motor.Target == 'STN'], _motor[_motor.Target == 'GPi']

print(f'\nPredictor screen (>= {MIN_NONZERO} patients with non-zero overlap):')
STN_PREDICTORS_L   = drop_unusable(STN_PREDICTORS_L,   _stn, 'STN L')
STN_PREDICTORS_R   = drop_unusable(STN_PREDICTORS_R,   _stn, 'STN R')
STN_PREDICTORS_AVG = drop_unusable(STN_PREDICTORS_AVG, _stn, 'STN avg')
GPi_PREDICTORS_L   = drop_unusable(GPi_PREDICTORS_L,   _gpi, 'GPi L')
GPi_PREDICTORS_R   = drop_unusable(GPi_PREDICTORS_R,   _gpi, 'GPi R')
GPi_PREDICTORS_AVG = drop_unusable(GPi_PREDICTORS_AVG, _gpi, 'GPi avg')

In [ ]:
# split by target
df_STN = df[df['Target'] == 'STN'].copy().reset_index(drop=True)
df_GPi = df[df['Target'] == 'GPi'].copy().reset_index(drop=True)

print(f'STN patients: n={len(df_STN)}')
print(f'  Laterality: {df_STN["Laterality"].value_counts().to_dict()}')
print(f'\nGPi patients: n={len(df_GPi)}')
print(f'  Laterality: {df_GPi["Laterality"].value_counts().to_dict()}')

In [ ]:
# union of predictors relevant to the group: L-only -> L+avg, R-only -> R+avg, Bi -> L+R+avg
def get_predictors(df_subset, target):
    lats = df_subset['Laterality'].unique()
    cols = set()
    
    if target == 'STN':
        L, R, AVG = STN_PREDICTORS_L, STN_PREDICTORS_R, STN_PREDICTORS_AVG
    else:
        L, R, AVG = GPi_PREDICTORS_L, GPi_PREDICTORS_R, GPi_PREDICTORS_AVG
    
    for lat in lats:
        if lat in ('L', 'Bi'):
            cols.update(L)
        if lat in ('R', 'Bi'):
            cols.update(R)
        cols.update(AVG)
    
    return [c for c in sorted(cols) if c in df_subset.columns]

stn_predictors = get_predictors(df_STN, 'STN')
gpi_predictors = get_predictors(df_GPi, 'GPi')

print(f'STN predictors ({len(stn_predictors)}): {stn_predictors}')
print(f'\nGPi predictors ({len(gpi_predictors)}): {gpi_predictors}')

## 2. descriptive statistics

In [ ]:
print('=== OUTCOME DESCRIPTIVES ===')
for grp_name, grp_df in [('STN', df_STN), ('GPi', df_GPi)]:
    print(f'\n--- {grp_name} (n={len(grp_df)}) ---')
    desc = grp_df[OUTCOMES].describe().T[['count', 'mean', 'std', '50%', 'min', 'max']]
    desc.columns = ['n', 'mean', 'sd', 'median', 'min', 'max']
    print(desc.round(2))

In [ ]:
# outcome distributions
n_out = len(OUTCOMES)
fig, axes = plt.subplots(2, n_out, figsize=(4 * n_out, 8), squeeze=False)
groups = [('STN', df_STN, '#2196F3'), ('GPi', df_GPi, '#FF9800')]

for row_i, (grp_name, grp_df, color) in enumerate(groups):
    for col_i, outcome in enumerate(OUTCOMES):
        ax = axes[row_i, col_i]
        data = grp_df[outcome].dropna()
        ax.hist(data, bins=15, color=color, alpha=0.7, edgecolor='white')
        ax.axvline(data.mean(), color='black', linestyle='--', linewidth=1.5,
                   label=f'Mean={data.mean():.1f}')
        star = '*' if outcome in PRIMARY else ''
        ax.set_title(f'{grp_name}{star}\n{OUTCOME_LABELS[outcome]}', fontsize=9, fontweight='bold')
        ax.set_xlabel('Δ MDS-UPDRS III (%)')
        ax.set_ylabel('Count')
        ax.legend(fontsize=8)

plt.suptitle('Outcome distributions by target   (* = primary endpoint)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 3. normality testing

Shapiro-Wilk on outcomes and predictors. both Pearson and Spearman are reported regardless.

In [ ]:
def normality_table(df_subset, columns, label):
    results = []
    for col in columns:
        data = df_subset[col].dropna()
        if len(data) < 3:
            continue
        stat, p = shapiro(data)
        results.append({
            'Variable': col,
            'n': len(data),
            'W': round(stat, 4),
            'p': round(p, 4),
            'Normal (p>0.05)': '✓' if p > 0.05 else '✗'
        })
    return pd.DataFrame(results)

print('=== NORMALITY: STN Outcomes ===')
norm_stn_out = normality_table(df_STN, OUTCOMES, 'STN')
display(norm_stn_out)

print('\n=== NORMALITY: GPi Outcomes ===')
norm_gpi_out = normality_table(df_GPi, OUTCOMES, 'GPi')
display(norm_gpi_out)

print('\n=== NORMALITY: STN Predictors ===')
norm_stn_pred = normality_table(df_STN, stn_predictors, 'STN')
display(norm_stn_pred)

print('\n=== NORMALITY: GPi Predictors ===')
norm_gpi_pred = normality_table(df_GPi, gpi_predictors, 'GPi')
display(norm_gpi_pred)

## 4. correlation analysis

Pearson and Spearman for each outcome × predictor pair. p-values are uncorrected (exploratory).

In [ ]:
def compute_correlations(df_subset, predictors, outcomes, group_label, alpha=0.05):
    all_results = []
    
    for outcome in outcomes:
        y = df_subset[outcome].dropna()
        
        for pred in predictors:
            x = df_subset[pred]
            combined = pd.concat([x, y], axis=1).dropna()
            if len(combined) < 5:
                continue
            x_clean, y_clean = combined.iloc[:, 0], combined.iloc[:, 1]
            
            r_p, p_p = pearsonr(x_clean, y_clean)
            r_s, p_s = spearmanr(x_clean, y_clean)
            
            all_results.append({'Predictor': pred, 'r': r_p, 'p': p_p,
                                  'sig': '*' if p_p < alpha else '', 'n': len(combined),
                                  'Method': 'Pearson', 'Outcome': outcome, 'Group': group_label})
            all_results.append({'Predictor': pred, 'r': r_s, 'p': p_s,
                                  'sig': '*' if p_s < alpha else '', 'n': len(combined),
                                  'Method': 'Spearman', 'Outcome': outcome, 'Group': group_label})
    
    return pd.DataFrame(all_results)

corr_STN = compute_correlations(df_STN, stn_predictors, OUTCOMES, 'STN')
corr_GPi = compute_correlations(df_GPi, gpi_predictors, OUTCOMES, 'GPi')

print(f'STN correlations computed: {len(corr_STN)} tests')
print(f'GPi correlations computed: {len(corr_GPi)} tests')

In [ ]:
def show_significant_correlations(corr_df, group_label, method='Spearman'):
    sub = corr_df[(corr_df['Method'] == method)].copy()
    sub = sub.sort_values(['Outcome', 'p'])
    n_col = 'n (nonzero)' if 'n (nonzero)' in sub.columns else 'n'
    sig = sub[sub['sig'] == '*'][['Group', 'Outcome', 'Predictor', 'r', 'p', n_col]]
    sig = sig.rename(columns={'r': 'r (corr)'})
    sig['r (corr)'] = sig['r (corr)'].round(3)
    sig['p'] = sig['p'].round(4)
    print(f'\n=== {group_label} {method}: Significant correlations (p<0.05) ===')
    if len(sig) == 0:
        print('  No significant correlations found.')
    else:
        display(sig)

for method in ['Pearson', 'Spearman']:
    show_significant_correlations(corr_STN, 'STN', method)
    show_significant_correlations(corr_GPi, 'GPi', method)

## 5. correlation heatmaps

In [ ]:
def plot_corr_heatmap(df_subset, predictors, outcomes, group_label, method='Spearman'):
    # one heatmap per side
    groups = [('L', [p for p in predictors if p.startswith('L_')]),
              ('R', [p for p in predictors if p.startswith('R_')]),
              ('avg', [p for p in predictors if p.startswith('avg_')])]

    for side, preds in groups:
        if not preds:
            continue
        r_matrix = pd.DataFrame(index=preds, columns=outcomes, dtype=float)
        p_matrix = pd.DataFrame(index=preds, columns=outcomes, dtype=float)

        for outcome in outcomes:
            for pred in preds:
                combined = df_subset[[pred, outcome]].dropna()
                if len(combined) < 5:
                    r_matrix.loc[pred, outcome] = np.nan
                    p_matrix.loc[pred, outcome] = np.nan
                    continue
                if method == 'Spearman':
                    r, p = spearmanr(combined[pred], combined[outcome])
                else:
                    r, p = pearsonr(combined[pred], combined[outcome])
                r_matrix.loc[pred, outcome] = r
                p_matrix.loc[pred, outcome] = p

        annot = pd.DataFrame(index=preds, columns=outcomes, dtype=str)
        for pred in preds:
            for outcome in outcomes:
                r = r_matrix.loc[pred, outcome]
                p_raw = p_matrix.loc[pred, outcome]
                if np.isnan(r):
                    annot.loc[pred, outcome] = 'N/A'
                else:
                    star = '*' if p_raw < 0.05 else ''
                    annot.loc[pred, outcome] = f'{r:.2f}{star}'

        row_labels = [clean_label(p) for p in preds]
        fig, ax = plt.subplots(figsize=(max(8, len(outcomes)*1.8), max(4, len(preds)*0.55)))
        # red = worse, blue = better
        sns.heatmap(r_matrix.astype(float), annot=annot, fmt='', cmap='RdBu',
                    center=0, vmin=-1, vmax=1, linewidths=0.5,
                    xticklabels=[OUTCOME_LABELS.get(o, o) for o in outcomes],
                    yticklabels=row_labels,
                    ax=ax, annot_kws={'size': 9})
        title = f'{group_label} {side} {method} Correlations\n* p<0.05   (red = worse outcome, blue = better)'
        ax.set_title(title, fontsize=11, fontweight='bold')
        ax.set_ylabel('Predictor')
        ax.set_xlabel('Outcome')
        plt.xticks(rotation=30, ha='right')
        plt.tight_layout()
        fname = f'heatmap_{group_label}_{side}_{method}'
        save_fig(fig, fname)
        plt.show()

for method in ['Pearson', 'Spearman']:
    plot_corr_heatmap(df_STN, stn_predictors, OUTCOMES, 'STN', method)
    plot_corr_heatmap(df_GPi, gpi_predictors, OUTCOMES, 'GPi', method)

## 6. scatter plots of significant correlations

In [ ]:
def plot_scatter_grid(df_subset, predictors, outcomes, group_label, method='Spearman', p_thresh=0.05, suffix=''):
    groups = [('L', [p for p in predictors if p.startswith('L_')]),
              ('R', [p for p in predictors if p.startswith('R_')]),
              ('avg', [p for p in predictors if p.startswith('avg_')])]

    for side, preds in groups:
        if not preds:
            continue
        sig_pairs = []
        for outcome in outcomes:
            for pred in preds:
                combined = df_subset[[pred, outcome]].dropna()
                if len(combined) < 5:
                    continue
                if method == 'Spearman':
                    r, p = spearmanr(combined[pred], combined[outcome])
                else:
                    r, p = pearsonr(combined[pred], combined[outcome])
                if p < p_thresh:
                    sig_pairs.append((pred, outcome, r, p, combined))

        if not sig_pairs:
            print(f'  No significant pairs for {group_label} {side} ({method}).')
            continue

        n_plots = len(sig_pairs)
        ncols = min(4, n_plots)
        nrows = int(np.ceil(n_plots / ncols))
        fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 4.5*nrows))
        axes = np.array(axes).flatten() if n_plots > 1 else [axes]

        for i, (pred, outcome, r, p, combined) in enumerate(sig_pairs):
            ax = axes[i]
            color = '#2196F3' if 'STN' in group_label else '#FF9800'
            ax.scatter(combined[pred], combined[outcome], alpha=0.7, edgecolors='white', s=60, color=color)
            m, b = np.polyfit(combined[pred], combined[outcome], 1)
            x_line = np.linspace(combined[pred].min(), combined[pred].max(), 100)
            ax.plot(x_line, m*x_line + b, 'r--', linewidth=1.5)
            ax.set_xlabel(clean_label(pred), fontsize=9)
            ax.set_ylabel(f'\u0394 {outcome}', fontsize=9)
            ax.set_title(f'{outcome} ~ {clean_label(pred)}\n{method} r={r:.3f}, p={p:.4f}', fontsize=9, fontweight='bold')

        for j in range(i+1, len(axes)):
            axes[j].set_visible(False)

        suptitle = f'{group_label} {side} Significant Correlations ({method}, p<{p_thresh})'
        if suffix:
            suptitle += f' [{suffix}]'
        plt.suptitle(suptitle, fontsize=12, fontweight='bold')
        plt.tight_layout()
        fname = f'scatter_{group_label}_{side}_{method}'
        if suffix:
            fname += f'_{suffix}'
        save_fig(fig, fname)
        plt.show()

for method in ['Pearson', 'Spearman']:
    plot_scatter_grid(df_STN, stn_predictors, OUTCOMES, 'STN', method)
    plot_scatter_grid(df_GPi, gpi_predictors, OUTCOMES, 'GPi', method)

## 7. STN vs GPi

compare each outcome between STN and GPi groups.

In [ ]:
print('=== STN vs GPi: Outcome Comparisons ===')
group_comp_results = []

for outcome in OUTCOMES:
    stn_vals = df_STN[outcome].dropna()
    gpi_vals = df_GPi[outcome].dropna()
    
    # normality
    _, p_norm_stn = shapiro(stn_vals) if len(stn_vals) >= 3 else (None, 0)
    _, p_norm_gpi = shapiro(gpi_vals) if len(gpi_vals) >= 3 else (None, 0)
    
    if p_norm_stn > 0.05 and p_norm_gpi > 0.05:
        stat, p = stats.ttest_ind(stn_vals, gpi_vals, equal_var=False)
        test_name = "Welch's t-test"
    else:
        stat, p = stats.mannwhitneyu(stn_vals, gpi_vals, alternative='two-sided')
        test_name = 'Mann-Whitney U'
    
    group_comp_results.append({
        'Outcome': outcome,
        'STN mean±SD': f'{stn_vals.mean():.2f} ± {stn_vals.std():.2f}',
        'GPi mean±SD': f'{gpi_vals.mean():.2f} ± {gpi_vals.std():.2f}',
        'Test': test_name,
        'Statistic': round(stat, 3),
        'p': round(p, 4),
        'Sig': '*' if p < 0.05 else ''
    })

display(pd.DataFrame(group_comp_results))

In [ ]:
n_out = len(OUTCOMES)
fig, axes = plt.subplots(1, n_out, figsize=(4.5 * n_out, 6), squeeze=False)
axes = axes[0]

for i, outcome in enumerate(OUTCOMES):
    ax = axes[i]
    plot_data = [
        df_STN[outcome].dropna().values,
        df_GPi[outcome].dropna().values
    ]
    bp = ax.boxplot(plot_data, labels=['STN', 'GPi'], patch_artist=True, notch=False,
                    medianprops=dict(color='black', linewidth=2))
    colors = ['#2196F3', '#FF9800']
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)

    for j, (vals, color) in enumerate(zip(plot_data, colors)):
        x_jitter = np.random.normal(j+1, 0.06, size=len(vals))
        ax.scatter(x_jitter, vals, alpha=0.5, color=color, s=30, zorder=3)

    p_row = group_comp_results[i]
    p_val = p_row['p']
    p_text = f'p={p_val:.3f}' + ('*' if p_val < 0.05 else '')
    y_max = max([v for vals in plot_data for v in vals])
    ax.text(1.5, y_max * 1.05, p_text, ha='center', fontsize=10, fontweight='bold')
    ax.plot([1, 2], [y_max * 1.03, y_max * 1.03], 'k-', linewidth=1)
    star = ' *' if outcome in PRIMARY else ''
    ax.set_title(f'{OUTCOME_LABELS[outcome]}{star}', fontweight='bold', fontsize=10)
    ax.set_ylabel('\u0394 MDS-UPDRS III (%)')
    ax.axhline(0, color='gray', linestyle=':', linewidth=1)

plt.suptitle('STN vs GPi: outcome comparisons   (* = primary endpoint)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
save_fig(fig, 'group_comparison_STN_vs_GPi')
plt.show()

## 8. simple linear regression (unadjusted)

univariate OLS for each predictor × outcome pair.

In [ ]:
def simple_regression_table(df_subset, predictors, outcomes, group_label, alpha=0.05):
    all_rows = []
    
    for outcome in outcomes:
        for pred in predictors:
            combined = df_subset[[pred, outcome, 'Age', 'Sex']].dropna()
            if len(combined) < 5:
                continue
            try:
                model = smf.ols(f'Q("{outcome}") ~ Q("{pred}")', data=combined).fit()
                coef = model.params[f'Q("{pred}")']
                ci_lo, ci_hi = model.conf_int().loc[f'Q("{pred}")'].values
                p = model.pvalues[f'Q("{pred}")']
                r2 = model.rsquared
                all_rows.append({'Predictor': pred, 'β': coef, 'CI_lo': ci_lo, 'CI_hi': ci_hi,
                                  'p': p, 'sig': '*' if p < alpha else '', 'R²': r2,
                                  'n': len(combined), 'Outcome': outcome, 'Group': group_label})
            except Exception:
                continue
    
    result_df = pd.DataFrame(all_rows)
    if len(result_df) > 0:
        result_df = result_df.round({'β': 4, 'CI_lo': 4, 'CI_hi': 4, 'p': 4, 'R²': 3})
    return result_df

reg_STN = simple_regression_table(df_STN, stn_predictors, OUTCOMES, 'STN')
reg_GPi = simple_regression_table(df_GPi, gpi_predictors, OUTCOMES, 'GPi')

for outcome in OUTCOMES:
    print(f'\n=== STN Simple Regression {outcome} ===')
    sub = reg_STN[reg_STN['Outcome'] == outcome][['Predictor','β','CI_lo','CI_hi','R²','p','sig','n']]
    display(sub.sort_values('p').reset_index(drop=True))

for outcome in OUTCOMES:
    print(f'\n=== GPi Simple Regression {outcome} ===')
    sub = reg_GPi[reg_GPi['Outcome'] == outcome][['Predictor','β','CI_lo','CI_hi','R²','p','sig','n']]
    display(sub.sort_values('p').reset_index(drop=True))

## 9. adjusted linear regression (age + sex + laterality)

In [ ]:
def adjusted_regression_table(df_subset, predictors, outcomes, group_label, alpha=0.05):
    df_mod = df_subset.copy()
    if df_mod['Sex'].dtype == object:
        df_mod['Sex_bin'] = (df_mod['Sex'].str.upper().str.strip() == 'M').astype(int)
    else:
        df_mod['Sex_bin'] = df_mod['Sex']
    df_mod['Lat_bin'] = (df_mod['Laterality'].str.upper().str.strip() == 'BI').astype(int)
    
    all_rows = []
    
    for outcome in outcomes:
        for pred in predictors:
            combined = df_mod[[pred, outcome, 'Age', 'Sex_bin', 'Lat_bin']].dropna()
            if len(combined) < 8:
                continue
            try:
                formula = f'Q("{outcome}") ~ Q("{pred}") + Age + Sex_bin + Lat_bin'
                model = smf.ols(formula, data=combined).fit()
                coef = model.params[f'Q("{pred}")']
                ci_lo, ci_hi = model.conf_int().loc[f'Q("{pred}")'].values
                p = model.pvalues[f'Q("{pred}")']
                r2_adj = model.rsquared_adj
                all_rows.append({'Predictor': pred, 'β (adj)': coef, 'CI_lo': ci_lo, 'CI_hi': ci_hi,
                                  'p': p, 'sig': '*' if p < alpha else '', 'Adj.R²': r2_adj,
                                  'n': len(combined), 'Outcome': outcome, 'Group': group_label})
            except Exception:
                continue
    
    result_df = pd.DataFrame(all_rows)
    if len(result_df) > 0:
        result_df = result_df.round({'β (adj)': 4, 'CI_lo': 4, 'CI_hi': 4, 'p': 4, 'Adj.R²': 3})
    return result_df

adj_reg_STN = adjusted_regression_table(df_STN, stn_predictors, OUTCOMES, 'STN')
adj_reg_GPi = adjusted_regression_table(df_GPi, gpi_predictors, OUTCOMES, 'GPi')

for outcome in OUTCOMES:
    print(f'\n=== STN Adjusted Regression (Age+Sex+Lat) {outcome} ===')
    sub = adj_reg_STN[adj_reg_STN['Outcome'] == outcome][['Predictor','β (adj)','CI_lo','CI_hi','Adj.R²','p','sig','n']]
    display(sub.sort_values('p').reset_index(drop=True))

for outcome in OUTCOMES:
    print(f'\n=== GPi Adjusted Regression (Age+Sex+Lat) {outcome} ===')
    sub = adj_reg_GPi[adj_reg_GPi['Outcome'] == outcome][['Predictor','β (adj)','CI_lo','CI_hi','Adj.R²','p','sig','n']]
    display(sub.sort_values('p').reset_index(drop=True))

## 10. regression coefficient plots

In [ ]:
def plot_coef_forest(reg_df, group_label, adjusted=False):
    beta_col = 'β (adj)' if adjusted else 'β'
    if beta_col not in reg_df.columns:
        return

    label = 'Adjusted' if adjusted else 'Unadjusted'
    color_sig = '#D32F2F'
    color_ns  = '#90A4AE'

    for side in ['L', 'R', 'avg']:
        if side == 'avg':
            side_df = reg_df[reg_df['Predictor'].str.startswith('avg')]
        else:
            side_df = reg_df[reg_df['Predictor'].str.startswith(side + '_')]
        if side_df.empty:
            continue

        outcomes = side_df['Outcome'].unique()
        fig, axes = plt.subplots(1, len(outcomes), figsize=(5.5*len(outcomes), max(4, side_df['Predictor'].nunique()*0.5)))
        if len(outcomes) == 1:
            axes = [axes]

        for ax, outcome in zip(axes, outcomes):
            sub = side_df[side_df['Outcome'] == outcome].sort_values(beta_col)
            y_pos = range(len(sub))
            colors = [color_sig if s == '*' else color_ns for s in sub['sig']]
            ax.barh(list(y_pos), sub[beta_col].values, xerr=[
                sub[beta_col].values - sub['CI_lo'].values,
                sub['CI_hi'].values - sub[beta_col].values
            ], color=colors, alpha=0.75, height=0.6, capsize=3, ecolor='gray')
            ax.axvline(0, color='black', linestyle='--', linewidth=1)
            ax.set_yticks(list(y_pos))
            ax.set_yticklabels([clean_label(p) for p in sub['Predictor'].values], fontsize=8)
            ax.set_xlabel('β coefficient', fontsize=9)
            ax.set_title(f'{outcome}\n{label}', fontweight='bold', fontsize=10)
            legend_elements = [Patch(facecolor=color_sig, alpha=0.75, label='p<0.05'),
                               Patch(facecolor=color_ns, alpha=0.75, label='n.s.')]
            ax.legend(handles=legend_elements, fontsize=8, loc='lower right')

        plt.suptitle(f'{group_label} {side} {label} Regression Coefficients (95% CI)', fontsize=12, fontweight='bold')
        plt.tight_layout()
        fname = f'forest_{group_label}_{side}_{"adj" if adjusted else "unadj"}'
        save_fig(fig, fname)
        plt.show()

plot_coef_forest(reg_STN, 'STN', adjusted=False)
plot_coef_forest(reg_GPi, 'GPi', adjusted=False)
plot_coef_forest(adj_reg_STN, 'STN', adjusted=True)
plot_coef_forest(adj_reg_GPi, 'GPi', adjusted=True)

## 11. multivariate regression

all predictors together (z-scored) + age + sex + laterality. likely underpowered at this n, exploratory.

In [ ]:
def multivariate_regression(df_subset, predictors, outcomes, group_label):
    df_mod = df_subset.copy()
    if df_mod['Sex'].dtype == object:
        df_mod['Sex_bin'] = (df_mod['Sex'].str.upper().str.strip() == 'M').astype(int)
    else:
        df_mod['Sex_bin'] = df_mod['Sex']
    df_mod['Lat_bin'] = (df_mod['Laterality'].str.upper().str.strip() == 'BI').astype(int)
    
    # z-score predictors
    scaler = StandardScaler()
    pred_cols_clean = [p for p in predictors if p in df_mod.columns]
    df_mod[pred_cols_clean] = scaler.fit_transform(df_mod[pred_cols_clean].fillna(df_mod[pred_cols_clean].mean()))
    
    for outcome in outcomes:
        needed = pred_cols_clean + [outcome, 'Age', 'Sex_bin', 'Lat_bin']
        combined = df_mod[needed].dropna()
        
        min_n = len(pred_cols_clean) + 3 + 5
        if len(combined) < min_n:
            print(f'\n{group_label} {outcome}: Insufficient n ({len(combined)}) for {len(pred_cols_clean)}-predictor model. Skipping.')
            continue
        
        try:
            safe_preds = [f'Q("{p}")' for p in pred_cols_clean]
            formula = f'Q("{outcome}") ~ {" + ".join(safe_preds)} + Age + Sex_bin + Lat_bin'
            model = smf.ols(formula, data=combined).fit()
            
            print(f'\n========== {group_label} Multivariate Model: {outcome} ==========')
            print(f'n={len(combined)}, R²={model.rsquared:.3f}, Adj.R²={model.rsquared_adj:.3f}, F-test p={model.f_pvalue:.4f}')
            
            summary_df = pd.DataFrame({
                'β (std)': model.params,
                'SE': model.bse,
                'CI_lo': model.conf_int().iloc[:, 0],
                'CI_hi': model.conf_int().iloc[:, 1],
                'p': model.pvalues
            }).round(4)
            summary_df['sig'] = summary_df['p'].apply(lambda x: '*' if x < 0.05 else '')
            display(summary_df)
        except Exception as e:
            print(f'  Model failed: {e}')

multivariate_regression(df_STN, stn_predictors, OUTCOMES, 'STN')
multivariate_regression(df_GPi, gpi_predictors, OUTCOMES, 'GPi')

## 12. summary of significant results

nominally significant (p<0.05) unadjusted and adjusted results for both groups.

In [ ]:
print('SIGNIFICANT RESULTS (p<0.05)')

for label, df_res, beta_col in [
    ('STN Unadjusted', reg_STN, 'β'),
    ('GPi Unadjusted', reg_GPi, 'β'),
    ('STN Adjusted (Age+Sex+Lat)', adj_reg_STN, 'β (adj)'),
    ('GPi Adjusted (Age+Sex+Lat)', adj_reg_GPi, 'β (adj)'),
]:
    if df_res is None or len(df_res) == 0:
        continue
    sig = df_res[df_res['sig'] == '*'][['Outcome','Predictor', beta_col,'CI_lo','CI_hi','p','n']].copy()
    sig = sig.sort_values(['Outcome','p']).reset_index(drop=True)
    print(f'\n--- {label} ---')
    if len(sig) == 0:
        print('  None')
    else:
        display(sig)

print('\n* = p<0.05 (uncorrected, exploratory)')

# part 2. non-zero overlap analyses

same pipeline, excluding patients with zero overlap for each predictor. asks whether, among patients with any stimulation of a region, more overlap predicts better outcome. n varies per test (`n (nonzero)`).

In [ ]:
# zero-overlap counts per predictor
print('=== Zero-overlap counts per STN predictor ===')
for p in stn_predictors:
    n_zero = (df_STN[p] == 0).sum()
    n_total = df_STN[p].notna().sum()
    print(f'  {p}: {n_zero}/{n_total} zeros ({100*n_zero/n_total:.0f}%)')

print('\n=== Zero-overlap counts per GPi predictor ===')
for p in gpi_predictors:
    n_zero = (df_GPi[p] == 0).sum()
    n_total = df_GPi[p].notna().sum()
    print(f'  {p}: {n_zero}/{n_total} zeros ({100*n_zero/n_total:.0f}%)')

## 13. non-zero correlations

In [ ]:
def compute_correlations_nonzero(df_subset, predictors, outcomes, group_label, alpha=0.05):
    all_results = []
    
    for outcome in outcomes:
        for pred in predictors:
            combined = df_subset[[pred, outcome]].dropna()
            combined = combined[combined[pred] != 0]
            if len(combined) < 5:
                continue
            x_clean, y_clean = combined[pred], combined[outcome]
            
            r_p, p_p = pearsonr(x_clean, y_clean)
            r_s, p_s = spearmanr(x_clean, y_clean)
            
            all_results.append({'Predictor': pred, 'r': r_p, 'p': p_p,
                                  'sig': '*' if p_p < alpha else '',
                                  'n (nonzero)': len(combined), 'Method': 'Pearson',
                                  'Outcome': outcome, 'Group': group_label})
            all_results.append({'Predictor': pred, 'r': r_s, 'p': p_s,
                                  'sig': '*' if p_s < alpha else '',
                                  'n (nonzero)': len(combined), 'Method': 'Spearman',
                                  'Outcome': outcome, 'Group': group_label})
    
    return pd.DataFrame(all_results)

corr_STN_nz = compute_correlations_nonzero(df_STN, stn_predictors, OUTCOMES, 'STN')
corr_GPi_nz = compute_correlations_nonzero(df_GPi, gpi_predictors, OUTCOMES, 'GPi')

for method in ['Pearson', 'Spearman']:
    show_significant_correlations(corr_STN_nz, 'STN [non-zero]', method)
    show_significant_correlations(corr_GPi_nz, 'GPi [non-zero]', method)

## 14. non-zero correlation heatmaps

In [ ]:
def plot_corr_heatmap_nonzero(df_subset, predictors, outcomes, group_label, method='Spearman'):
    groups = [('L', [p for p in predictors if p.startswith('L_')]),
              ('R', [p for p in predictors if p.startswith('R_')]),
              ('avg', [p for p in predictors if p.startswith('avg_')])]

    for side, preds in groups:
        if not preds:
            continue
        r_matrix = pd.DataFrame(index=preds, columns=outcomes, dtype=float)
        p_matrix = pd.DataFrame(index=preds, columns=outcomes, dtype=float)
        n_matrix = pd.DataFrame(index=preds, columns=outcomes, dtype=float)

        for outcome in outcomes:
            for pred in preds:
                combined = df_subset[[pred, outcome]].dropna()
                combined = combined[combined[pred] != 0]
                if len(combined) < 5:
                    r_matrix.loc[pred, outcome] = np.nan
                    p_matrix.loc[pred, outcome] = np.nan
                    n_matrix.loc[pred, outcome] = len(combined)
                    continue
                if method == 'Spearman':
                    r, p = spearmanr(combined[pred], combined[outcome])
                else:
                    r, p = pearsonr(combined[pred], combined[outcome])
                r_matrix.loc[pred, outcome] = r
                p_matrix.loc[pred, outcome] = p
                n_matrix.loc[pred, outcome] = len(combined)

        annot = pd.DataFrame(index=preds, columns=outcomes, dtype=str)
        for pred in preds:
            for outcome in outcomes:
                r = r_matrix.loc[pred, outcome]
                p_raw = p_matrix.loc[pred, outcome]
                n = int(n_matrix.loc[pred, outcome]) if not np.isnan(n_matrix.loc[pred, outcome]) else 0
                if np.isnan(r):
                    annot.loc[pred, outcome] = f'n={n}'
                else:
                    star = '*' if p_raw < 0.05 else ''
                    annot.loc[pred, outcome] = f'{r:.2f}{star}\nn={n}'

        row_labels = [clean_label(p) for p in preds]
        fig, ax = plt.subplots(figsize=(max(8, len(outcomes)*2), max(4, len(preds)*0.65)))
        sns.heatmap(r_matrix.astype(float), annot=annot, fmt='', cmap='RdBu',
                    center=0, vmin=-1, vmax=1, linewidths=0.5,
                    xticklabels=[OUTCOME_LABELS.get(o, o) for o in outcomes],
                    yticklabels=row_labels,
                    ax=ax, annot_kws={'size': 8})
        ax.set_title(f'{group_label} {side} {method} Correlations [NON-ZERO ONLY]\n* p<0.05   n = patients with >0 overlap   (red = worse, blue = better)',
                     fontsize=10, fontweight='bold')
        ax.set_ylabel('Predictor')
        ax.set_xlabel('Outcome')
        plt.xticks(rotation=30, ha='right')
        plt.tight_layout()
        fname = f'heatmap_nonzero_{group_label}_{side}_{method}'
        save_fig(fig, fname)
        plt.show()

for method in ['Pearson', 'Spearman']:
    plot_corr_heatmap_nonzero(df_STN, stn_predictors, OUTCOMES, 'STN', method)
    plot_corr_heatmap_nonzero(df_GPi, gpi_predictors, OUTCOMES, 'GPi', method)

## 15. non-zero scatter plots

In [ ]:
def plot_scatter_grid_nonzero(df_subset, predictors, outcomes, group_label, method='Spearman', p_thresh=0.05):
    groups = [('L', [p for p in predictors if p.startswith('L_')]),
              ('R', [p for p in predictors if p.startswith('R_')]),
              ('avg', [p for p in predictors if p.startswith('avg_')])]

    for side, preds in groups:
        if not preds:
            continue
        sig_pairs = []
        for outcome in outcomes:
            for pred in preds:
                combined = df_subset[[pred, outcome]].dropna()
                combined = combined[combined[pred] != 0]
                if len(combined) < 5:
                    continue
                if method == 'Spearman':
                    r, p = spearmanr(combined[pred], combined[outcome])
                else:
                    r, p = pearsonr(combined[pred], combined[outcome])
                if p < p_thresh:
                    sig_pairs.append((pred, outcome, r, p, combined))

        if not sig_pairs:
            print(f'  No significant pairs for {group_label} {side} ({method}) [non-zero].')
            continue

        n_plots = len(sig_pairs)
        ncols = min(4, n_plots)
        nrows = int(np.ceil(n_plots / ncols))
        fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 4.5*nrows))
        axes = np.array(axes).flatten() if n_plots > 1 else [axes]

        for i, (pred, outcome, r, p, combined) in enumerate(sig_pairs):
            ax = axes[i]
            color = '#1565C0' if 'STN' in group_label else '#E65100'
            ax.scatter(combined[pred], combined[outcome], alpha=0.7, edgecolors='white', s=60, color=color)
            m, b = np.polyfit(combined[pred], combined[outcome], 1)
            x_line = np.linspace(combined[pred].min(), combined[pred].max(), 100)
            ax.plot(x_line, m*x_line + b, 'r--', linewidth=1.5)
            ax.set_xlabel(clean_label(pred), fontsize=9)
            ax.set_ylabel(f'\u0394 {outcome}', fontsize=9)
            ax.set_title(f'{outcome} ~ {clean_label(pred)}\n{method} r={r:.3f}, p={p:.4f}  [n={len(combined)}, non-zero]',
                         fontsize=9, fontweight='bold')

        for j in range(i+1, len(axes)):
            axes[j].set_visible(False)

        plt.suptitle(f'{group_label} {side} Significant Correlations [{method}, non-zero, p<{p_thresh}]',
                     fontsize=12, fontweight='bold')
        plt.tight_layout()
        fname = f'scatter_nonzero_{group_label}_{side}_{method}'
        save_fig(fig, fname)
        plt.show()

for method in ['Pearson', 'Spearman']:
    plot_scatter_grid_nonzero(df_STN, stn_predictors, OUTCOMES, 'STN', method)
    plot_scatter_grid_nonzero(df_GPi, gpi_predictors, OUTCOMES, 'GPi', method)

## 16. non-zero simple regression (unadjusted)

In [ ]:
def simple_regression_nonzero(df_subset, predictors, outcomes, group_label, alpha=0.05):
    all_rows = []
    
    for outcome in outcomes:
        for pred in predictors:
            combined = df_subset[[pred, outcome]].dropna()
            combined = combined[combined[pred] != 0]
            if len(combined) < 5:
                continue
            try:
                model = smf.ols(f'Q("{outcome}") ~ Q("{pred}")', data=combined).fit()
                coef = model.params[f'Q("{pred}")']
                ci_lo, ci_hi = model.conf_int().loc[f'Q("{pred}")'].values
                p = model.pvalues[f'Q("{pred}")']
                r2 = model.rsquared
                all_rows.append({'Predictor': pred, 'β': coef, 'CI_lo': ci_lo, 'CI_hi': ci_hi,
                                  'p': p, 'sig': '*' if p < alpha else '', 'R²': r2,
                                  'n (nonzero)': len(combined), 'Outcome': outcome, 'Group': group_label})
            except Exception:
                continue
    
    result_df = pd.DataFrame(all_rows)
    if len(result_df) > 0:
        result_df = result_df.round({'β': 4, 'CI_lo': 4, 'CI_hi': 4, 'p': 4, 'R²': 3})
    return result_df

nz_reg_STN = simple_regression_nonzero(df_STN, stn_predictors, OUTCOMES, 'STN')
nz_reg_GPi = simple_regression_nonzero(df_GPi, gpi_predictors, OUTCOMES, 'GPi')

for outcome in OUTCOMES:
    print(f'\n=== STN Simple Regression [non-zero] {outcome} ===')
    sub = nz_reg_STN[nz_reg_STN['Outcome'] == outcome][['Predictor','β','CI_lo','CI_hi','R²','p','sig','n (nonzero)']]
    display(sub.sort_values('p').reset_index(drop=True))

for outcome in OUTCOMES:
    print(f'\n=== GPi Simple Regression [non-zero] {outcome} ===')
    sub = nz_reg_GPi[nz_reg_GPi['Outcome'] == outcome][['Predictor','β','CI_lo','CI_hi','R²','p','sig','n (nonzero)']]
    display(sub.sort_values('p').reset_index(drop=True))

## 17. non-zero adjusted regression (age + sex + laterality)

In [ ]:
def adjusted_regression_nonzero(df_subset, predictors, outcomes, group_label, alpha=0.05):
    df_mod = df_subset.copy()
    if df_mod['Sex'].dtype == object:
        df_mod['Sex_bin'] = (df_mod['Sex'].str.upper().str.strip() == 'M').astype(int)
    else:
        df_mod['Sex_bin'] = df_mod['Sex']
    df_mod['Lat_bin'] = (df_mod['Laterality'].str.upper().str.strip() == 'BI').astype(int)
    
    all_rows = []
    
    for outcome in outcomes:
        for pred in predictors:
            combined = df_mod[[pred, outcome, 'Age', 'Sex_bin', 'Lat_bin']].dropna()
            combined = combined[combined[pred] != 0]
            if len(combined) < 8:
                continue
            try:
                formula = f'Q("{outcome}") ~ Q("{pred}") + Age + Sex_bin + Lat_bin'
                model = smf.ols(formula, data=combined).fit()
                coef = model.params[f'Q("{pred}")']
                ci_lo, ci_hi = model.conf_int().loc[f'Q("{pred}")'].values
                p = model.pvalues[f'Q("{pred}")']
                r2_adj = model.rsquared_adj
                all_rows.append({'Predictor': pred, 'β (adj)': coef, 'CI_lo': ci_lo, 'CI_hi': ci_hi,
                                  'p': p, 'sig': '*' if p < alpha else '', 'Adj.R²': r2_adj,
                                  'n (nonzero)': len(combined), 'Outcome': outcome, 'Group': group_label})
            except Exception:
                continue
    
    result_df = pd.DataFrame(all_rows)
    if len(result_df) > 0:
        result_df = result_df.round({'β (adj)': 4, 'CI_lo': 4, 'CI_hi': 4, 'p': 4, 'Adj.R²': 3})
    return result_df

nz_adj_reg_STN = adjusted_regression_nonzero(df_STN, stn_predictors, OUTCOMES, 'STN')
nz_adj_reg_GPi = adjusted_regression_nonzero(df_GPi, gpi_predictors, OUTCOMES, 'GPi')

for outcome in OUTCOMES:
    print(f'\n=== STN Adjusted Regression [non-zero] {outcome} ===')
    sub = nz_adj_reg_STN[nz_adj_reg_STN['Outcome'] == outcome][['Predictor','β (adj)','CI_lo','CI_hi','Adj.R²','p','sig','n (nonzero)']]
    display(sub.sort_values('p').reset_index(drop=True))

for outcome in OUTCOMES:
    print(f'\n=== GPi Adjusted Regression [non-zero] {outcome} ===')
    sub = nz_adj_reg_GPi[nz_adj_reg_GPi['Outcome'] == outcome][['Predictor','β (adj)','CI_lo','CI_hi','Adj.R²','p','sig','n (nonzero)']]
    display(sub.sort_values('p').reset_index(drop=True))

## 18. full sample vs non-zero

In [ ]:
print('FULL SAMPLE vs NON-ZERO, nominally significant (p<0.05)')

comparison_pairs = [
    ('STN Unadjusted',         reg_STN,         'STN Unadjusted [non-zero]',         nz_reg_STN,         'β'),
    ('GPi Unadjusted',         reg_GPi,          'GPi Unadjusted [non-zero]',          nz_reg_GPi,          'β'),
    ('STN Adjusted (Age+Sex+Lat)', adj_reg_STN,      'STN Adjusted [non-zero]',            nz_adj_reg_STN,      'β (adj)'),
    ('GPi Adjusted (Age+Sex+Lat)', adj_reg_GPi,       'GPi Adjusted [non-zero]',            nz_adj_reg_GPi,       'β (adj)'),
]

for full_label, df_full, nz_label, df_nz, beta_col in comparison_pairs:
    if df_full is None or df_nz is None:
        continue
    
    full_sig = set(zip(df_full[df_full['sig']=='*']['Outcome'], df_full[df_full['sig']=='*']['Predictor']))
    nz_sig   = set(zip(df_nz[df_nz['sig']=='*']['Outcome'],   df_nz[df_nz['sig']=='*']['Predictor']))
    
    consistent  = full_sig & nz_sig
    full_only   = full_sig - nz_sig
    nz_only     = nz_sig - full_sig
    
    print(f'\n--- {full_label} ---')
    print(f'  Significant in BOTH:        {sorted(consistent) if consistent else "None"}')
    print(f'  Full-sample only (zeros may drive):  {sorted(full_only) if full_only else "None"}')
    print(f'  Non-zero only (zeros suppress):      {sorted(nz_only) if nz_only else "None"}')

print('\n* = p<0.05 (uncorrected, exploratory)')

## notes on interpretation

- **primary vs secondary endpoints**: `MDSUPDRS_G` is primary. the preop-referenced scores (`A`, `C`) also include disease progression, microlesion and medication effects.
- **predictor screening**: regions with < 5 patients with any overlap in the motor subgroup are excluded (left GPi occipital and the prefrontal GPi predictors). the screen uses only the predictor distribution.
- **direction**: positive values mean improvement (score decreased).
- **multiple comparisons**: p-values are uncorrected (exploratory, ~18-20 patients per target).
- **cross-hemisphere averages**: `avg_` predictors are derived here as (L + R) / 2 with a non-implanted hemisphere as 0, matching the ΔLEDD analysis.
- **non-zero analyses**: results significant in the full sample but not the non-zero sample may be driven by patients with no stimulation of that region.
- **RN**: excluded, near-zero overlap in most patients.
- **experimental unit**: the patient. left, right and average predictors are fitted in separate models.

## export results

In [ ]:
def tag(df, analysis, sample):
    d = df.copy()
    d.insert(0, 'Sample', sample)
    d.insert(0, 'Analysis', analysis)
    return d

frames = []

# correlations
for analysis, df_res in [
    ('Correlation_Full',    corr_STN),
    ('Correlation_Full',    corr_GPi),
    ('Correlation_NZ',      corr_STN_nz),
    ('Correlation_NZ',      corr_GPi_nz),
]:
    if df_res is not None and not df_res.empty:
        sample = 'full' if 'NZ' not in analysis else 'nonzero'
        frames.append(tag(df_res, analysis, sample))

# regressions
for analysis, df_res in [
    ('Regression_Unadjusted',    reg_STN),
    ('Regression_Unadjusted',    reg_GPi),
    ('Regression_Adjusted',      adj_reg_STN),
    ('Regression_Adjusted',      adj_reg_GPi),
    ('Regression_NZ_Unadjusted', nz_reg_STN),
    ('Regression_NZ_Unadjusted', nz_reg_GPi),
    ('Regression_NZ_Adjusted',   nz_adj_reg_STN),
    ('Regression_NZ_Adjusted',   nz_adj_reg_GPi),
]:
    if df_res is not None and not df_res.empty:
        sample = 'full' if 'NZ' not in analysis else 'nonzero'
        frames.append(tag(df_res, analysis, sample))

out = pd.concat(frames, ignore_index=True, sort=False)
out.to_csv(OUT_CSV, index=False)
print(f'saved {len(out)} rows to {OUT_CSV}')